# CF03 COPT — GPU NN Forward Pass
**ECE 410/510 Spring 2026**

3-layer FC network (40 -> 256 -> 128 -> 10) matching the keyword-spotting project.
Kernel: tiled GEMM + fused bias + ReLU, then softmax.

> Before running: **Runtime -> Change runtime type -> T4 GPU**

In [ ]:
# Confirm T4 GPU
!nvidia-smi | head -20
!nvcc --version

In [ ]:
# Upload nn_forward.cu via the Files panel on the left, then compile:
!nvcc -O2 -o nn_forward nn_forward.cu -lm
print('Compiled OK')

In [ ]:
# Run the forward pass benchmark
!./nn_forward

In [ ]:
# Roofline plot
# Fill in the values printed by ./nn_forward above

import numpy as np
import matplotlib.pyplot as plt

# T4 hardware
peak_compute = 8100   # GFLOPS FP32
peak_bw      = 300    # GB/s
ridge        = peak_compute / peak_bw   # 27 FLOP/byte

# Fill in from ./nn_forward output
gflops_gpu = 0.0    # <-- FILL IN
ai_gpu     = 0.0    # <-- FILL IN (Arithmetic Intensity printed by kernel)

# Reference: GEMM kernels from CLLM
gflops_naive = 316.09
gflops_tiled = 392.46
ai_gemm      = 170.67

# Roofline curve
ai_x     = np.logspace(-1, 4, 2000)
roofline = np.minimum(ai_x * peak_bw, peak_compute)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(ai_x, roofline, 'k-', linewidth=2.2, label='T4 Roofline')
ax.axvline(ridge, color='gray', linestyle='--', linewidth=1.2)
ax.text(ridge * 1.08, 12, f'Ridge\n{ridge:.0f} F/B', fontsize=8.5, color='gray')

# CLLM reference kernels
ax.plot(ai_gemm, gflops_naive, 'o', color='lightcoral', markersize=9, zorder=4,
        label=f'GEMM Naive -- {gflops_naive:.0f} GFLOP/s')
ax.plot(ai_gemm, gflops_tiled, 's', color='steelblue', markersize=9, zorder=4,
        label=f'GEMM Tiled -- {gflops_tiled:.0f} GFLOP/s')

# COPT: NN forward pass
ax.plot(ai_gpu, gflops_gpu, 'D', color='darkorange', markersize=11, zorder=5,
        label=f'NN Fwd Pass -- {gflops_gpu:.0f} GFLOP/s')
ax.annotate(f'NN Fwd\n{gflops_gpu:.0f} GFLOP/s',
            xy=(ai_gpu, gflops_gpu),
            xytext=(ai_gpu * 0.08, gflops_gpu * 3),
            fontsize=8.5, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange', lw=1.3))

ax.text(0.15, 40, 'Memory-Bound',  fontsize=9, color='gray', alpha=0.7)
ax.text(300,  40, 'Compute-Bound', fontsize=9, color='gray', alpha=0.7)

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=11)
ax.set_ylabel('Attainable Performance (GFLOPS)',   fontsize=11)
ax.set_title('CUDA NN Forward Pass Roofline -- NVIDIA T4 GPU\n'
             'FC(40-256-128-10), Batch=256, FP32 | Codefest 3 COPT',
             fontsize=11, fontweight='bold', pad=10)
ax.set_xlim(1e-1, 1e4)
ax.set_ylim(1e1,  1e5)
ax.grid(True, which='both', linestyle=':', linewidth=0.6, alpha=0.5)
ax.legend(fontsize=9, loc='upper left', framealpha=0.9)

plt.tight_layout()
plt.savefig('nn_roofline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved nn_roofline.png -- download and add to codefest/cf03/profiling/')